# Preparing data

Run from codes/day-2/. Synthetic household amounts are EUR per month. Attempt each exercise before its worked answer.

## The input before any changes

Income and consumption are EUR per month. Each ID identifies a synthetic household.

In [ ]:
import pandas as pd
raw = pd.read_csv("data/raw/households.csv")
clean = raw.copy()
print("Source rows:", len(raw))
print(clean.head(3).to_string(index=False))


## An exact duplicate

duplicated marks repeated rows after their first occurrence.

In [ ]:
tiny = pd.DataFrame({
    "id": ["A", "A", "B"],
    "income": [1000, 1000, 1500],
})
print("Duplicate rows:", tiny.duplicated().tolist())
print(tiny.drop_duplicates().to_string(index=False))


## The same ID can hide conflicting values

These A rows disagree. Removing one requires information beyond the ID.

In [ ]:
conflict = pd.DataFrame({
    "id": ["A", "A", "B"],
    "income": [1000, 1200, 1500],
})
print("Exact duplicates:", conflict.duplicated().sum())
print(conflict.loc[conflict["id"].duplicated(keep=False)]
      .to_string(index=False))


## Removing the repeated classroom record

The fixture has one repeated row and 200 distinct household records.

In [ ]:
print("Exact duplicates:", clean.duplicated().sum())
clean = clean.drop_duplicates().copy()
print("Rows after deduplication:", len(clean))
print("Unique IDs:", clean["household_id"].is_unique)


## Converting text to numbers

errors="coerce" converts unusable text to a missing numerical value.

In [ ]:
text_income = pd.Series(["1000", "missing", "", None])
numeric = pd.to_numeric(text_income, errors="coerce")
print(pd.DataFrame({
    "input": text_income, "numeric": numeric,
}).to_string(index=False))


## Converting the income column

Two incomes and three consumption amounts need attention.

In [ ]:
print("Before:", clean["income"].dtype)
clean["income"] = pd.to_numeric(clean["income"], errors="coerce")
print("After:", clean["income"].dtype)
print(clean[["income", "consumption"]].isna().sum().to_string())


## Zero income is an observed value

A missing value and a measured zero answer different questions.

In [ ]:
income = pd.Series([1000.0, 0.0, None])
print("Income:", income.tolist())
print("Missing:", income.isna().tolist())
print("Zero:", income.eq(0).tolist())


## Finding the incomplete records

any(axis=1) asks whether either required amount is missing within each row.

In [ ]:
required = ["income", "consumption"]
incomplete = clean[required].isna().any(axis=1)
cols = ["household_id", "income", "consumption"]
print(clean.loc[incomplete, cols].to_string(index=False))


## Exercise 1 · A complete-record sample

Keep observed zero incomes. Use dropna(subset=...) to express the sample rule.

```python
# Start with clean after deduplication and numeric conversion.
# Keep rows with BOTH income and consumption observed.
# Print counts before and after, and the zero-income count.
# Verify that raw still contains 201 rows.
```

In [ ]:
# Try the exercise here.


## Worked answer · Five exclusions, two zeros retained

This lesson uses the 195 complete monetary records for all later comparisons.

In [ ]:
before = len(clean)
clean = clean.dropna(subset=required).copy()
print("Before:", before, "After:", len(clean))
print("Zero incomes:", clean["income"].eq(0).sum())
print("Original rows:", len(raw))


## Saving for every retained household

Negative saving records consumption above current income.

In [ ]:
clean["saving"] = clean["income"] - clean["consumption"]
cols = ["household_id", "income", "consumption", "saving"]
print(clean[cols].head(4).to_string(index=False))
print("Negative saving:", clean["saving"].lt(0).sum())


## Income per person

Check the denominator: household size must be positive.

In [ ]:
clean["income_per_person"] = clean["income"] / clean["size"]
cols = ["household_id", "income", "size", "income_per_person"]
print(clean[cols].head(3).round(2).to_string(index=False))
print("Nonpositive sizes:", clean["size"].le(0).sum())


## Division by zero in a column

The second share is infinite. It has no useful saving-rate interpretation.

In [ ]:
example = pd.DataFrame({
    "income": [1000.0, 0.0],
    "consumption": [800.0, 200.0],
})
example["saving"] = example["income"] - example["consumption"]
example["share"] = example["saving"] / example["income"]
print(example.to_string(index=False))


## A denominator only where income is positive

where keeps values satisfying the condition and marks the others missing.

In [ ]:
income = pd.Series([1000.0, 0.0, 800.0])
denominator = income.where(income > 0)
print(pd.DataFrame({
    "income": income, "denominator": denominator,
}).to_string(index=False))


## Amounts remain defined when shares are missing

All 195 households have saving amounts. 193 have defined saving shares.

In [ ]:
denominator = clean["income"].where(clean["income"] > 0)
clean["saving_share"] = clean["saving"] / denominator
cols = ["household_id", "income", "saving", "saving_share"]
print(clean.loc[clean["income"].eq(0), cols].to_string(index=False))
print("Defined shares:", clean["saving_share"].count())


## Exercise 2 · Larger households with low income

Use the cleaned sample. Put each condition in parentheses before combining them.

```python
# Add negative_saving = saving < 0.
# Select size >= 3, income_per_person < 750,
# and negative_saving == True.
# Print household_id, size, income_per_person and saving.
# Show three rows, then the total number selected.
```

In [ ]:
# Try the exercise here.


## Worked answer · Three conditions



In [ ]:
clean["negative_saving"] = clean["saving"] < 0
mask = (
    (clean["size"] >= 3) & (clean["income_per_person"] < 750)
    & clean["negative_saving"]
)
cols = ["household_id", "size", "income_per_person", "saving"]
selected = clean.loc[mask, cols]
print(selected.head(3).round(2).to_string(index=False))
print("Selected:", len(selected))


## Checks that match the data definitions

Assertions stop the run when an assumed property fails.

In [ ]:
assert clean["household_id"].is_unique
assert clean["size"].gt(0).all()
assert clean[["income", "consumption"]].ge(0).all().all()
assert clean[required].notna().all().all()
print("Passed: IDs, sizes, money inputs and required values")


## Selecting suspicious rows for inspection

The mask identifies B and C for investigation.

In [ ]:
check = pd.DataFrame({
    "id": ["A", "B", "C"], "income": [1000, -100, 800],
    "size": [2, 1, 0],
})
bad = (check["income"] < 0) | (check["size"] <= 0)
print(check.loc[bad].to_string(index=False))


## Rounding a display keeps the underlying values

Round the table for readers after making calculations with the stored values.

In [ ]:
values = pd.Series([1000 / 3, 800 / 3])
print("Display:", values.round(1).tolist())
print("Stored:", values.tolist())
print("Mean from stored values:", values.mean())


## A preparation function



In [ ]:
def prepare_households(source):
    out = source.drop_duplicates().copy()
    out["income"] = pd.to_numeric(out["income"], errors="coerce")
    out = out.dropna(subset=["income", "consumption"]).copy()
    out["saving"] = out["income"] - out["consumption"]
    out["income_per_person"] = out["income"] / out["size"]
    denominator = out["income"].where(out["income"] > 0)
    out["saving_share"] = out["saving"] / denominator
    return out

prepared = prepare_households(raw)
print("Prepared households:", len(prepared))


## Checking a fresh preparation run

Reloading the original input prevents dependence on an earlier notebook state.

In [ ]:
fresh = prepare_households(pd.read_csv("data/raw/households.csv"))
assert len(fresh) == 195
assert fresh["household_id"].is_unique
assert fresh["saving_share"].count() == 193
print(fresh[["saving", "income_per_person"]].head(3).to_string(index=False))


## Exporting the prepared table

The export goes to outputs/. The original CSV stays in data/raw/.

In [ ]:
from pathlib import Path
Path("outputs").mkdir(exist_ok=True)
fresh.to_csv("outputs/prepared_households.csv", index=False)
check = pd.read_csv("outputs/prepared_households.csv")
print("Exported rows:", len(check))
print("Undefined shares:", check["saving_share"].isna().sum())
